In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql import types as t
from pyspark.sql.window import Window
from datetime import datetime
import logging        
from config import ROUTES, PipelineConfig  

In [0]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)

In [0]:
# Config 
BRONZE_PATH = "workspace.case_spark_cvm.bronze_registro_classe_cvm"
NOME_TABELA  = f"silver_registro_classe_cvm" 
SILVER_PATH = f"{ROUTES.TABLE_BASE}.{NOME_TABELA}"
DATA_PROC    = int(datetime.now().strftime("%Y%m%d"))

## CVM - Fundos Investimentos - Registros Classes

In [0]:
df_registro_classe_cvm = PipelineConfig.ler_ultima_particao(spark=spark, table_name=BRONZE_PATH, partition_col="data_processamento" )

### 1.1 tratemento silver

#### 1.1.1 Normalizando CNPJ

In [0]:
df_registro_classe_cvm = df_registro_classe_cvm.withColumn(
    "CNPJ_Classe",
    PipelineConfig.normalizar_cnpj("CNPJ_Classe")
)

#### 1.1.2 Retirando dados duplicados

A origem de dados da CVM é transacional e envia múltiplos registros para o mesmo ***CNPJ_Classe*** ao longo do tempo, refletindo a evolução do ciclo de vida do fundo (ex: atualizações de status ou encerramentos).

Para garantir a unicidade desta Tabela Dimensão e refletir a realidade mais atualizada do fundo, aplicamos uma lógica de desempate em duas etapas:
1. **Prioridade Temporal:** Mantemos exclusivamente o registro com a ***Data_Registro*** mais recente.
2. **Desempate por Status:** Caso existam múltiplos registros na mesma data, priorizamos a situação operacional (pesos maiores para *"Em Funcionamento Normal"* e *"Fase Pré-Operacional"*).

*Nota:* Duplicidades residuais que indicam erro na origem (ex: registros idênticos no mesmo dia) são capturadas e isoladas na tabela de Quarentena para posterior notificação.

In [0]:

# A função remover_duplicatas usa a ordem desc(). 
# Para que 'Normal' ganhe de 'Cancelado' no mesmo dia, damos a letra 'B' (maior) para Normal e 'A' para Cancelado.
df_registro_classe_cvm = df_registro_classe_cvm.withColumn(
    "_peso_status",
    f.when(f.col("Situacao").isin("Em Funcionamento Normal", "Fase Pré-Operacional"), f.lit("B"))
     .otherwise(f.lit("A"))
)

# Criamos uma super-chave. Ex: "2025-06-27_B" vence de "2025-06-27_A" e de "2024-01-01_B"
df_registro_classe_cvm = df_registro_classe_cvm.withColumn(
    "_ordem_desempate",
    f.concat_ws("_", f.col("Data_Registro"), f.col("_peso_status"))
)

# Espalhamos a Situação Vencedora para todas as linhas daquele CNPJ (necessário para o filtro da quarentena depois)
window_cnpj = Window.partitionBy("CNPJ_Classe")

df_registro_classe_cvm = df_registro_classe_cvm.withColumn(
    "_max_ordem_desempate", 
    f.max("_ordem_desempate").over(window_cnpj)
).withColumn(
    "_situacao_oficial",
    f.max(
        f.when(f.col("_ordem_desempate") == f.col("_max_ordem_desempate"), f.col("Situacao"))
    ).over(window_cnpj)
)


# Chamamos a função central do pipeline usando a nossa super-chave de desempate
df_registro_classe_cvm, df_todas_duplicadas = PipelineConfig.remover_duplicatas(
    df=df_registro_classe_cvm,
    chave_negocio=["CNPJ_Classe"],
    coluna_ordenacao="_ordem_desempate"
)

# O df_todas_duplicadas tem TUDO o que foi descartado. 
# Só é uma anomalia grave (digna de e-mail para a CVM) se a linha descartada tiver a MESMA situação da vencedora.
df_quarentena_real = (df_todas_duplicadas
    .filter(f.col("Situacao") == f.col("_situacao_oficial"))
    .withColumn("_motivo_quarentena", f.lit("Anomalia CVM: Múltiplos registros com status conflitante para o mesmo CNPJ"))
)

# Limpeza pesada: Removemos todas as colunas de controle da Silver e da Quarentena
colunas_sujeira = ["_peso_status", "_ordem_desempate", "_situacao_oficial"]
df_registro_classe_cvm = df_registro_classe_cvm.drop(*colunas_sujeira)
df_quarentena_real = df_quarentena_real.drop(*colunas_sujeira)

PipelineConfig.salvar_quarentena(
    spark=spark,
    df_quarentena=df_quarentena_real, 
    tabela_origem="bronze_registro_classe_cvm", 
    data_proc=DATA_PROC
)

#### 1.1.3 Retirando dados nulos de Colunas Cores

In [0]:
regras_qualidade = {
    "ID_Registro_Fundo": "not_null",   # Não pode ser vazio (Substitui o dropna)
    "ID_Registro_Classe": "not_null",  # Não pode ser vazio (Substitui o dropna)
    "CNPJ_Classe": "not_null",         # Não pode ser vazio (Substitui o dropna)
    "Data_Registro": "not_null",       # Não pode ser vazio (Substitui o dropna)
    "Situacao": "not_null",            # Não pode ser vazio (Substitui o dropna)
}

df_registro_classe_cvm, df_quarentena = PipelineConfig.aplicar_qualidade_e_separar(
    df=df_registro_classe_cvm,
    regras=regras_qualidade
    )

PipelineConfig.salvar_quarentena(
    spark=spark,
    df_quarentena=df_quarentena, 
    tabela_origem="bronze_registro_classe_cvm", 
    data_proc=DATA_PROC
)

#### 1.1.4 Tratamento do Tipo de Dado

In [0]:
# Dropando as colunas de metadados
df_registro_classe_cvm = df_registro_classe_cvm.drop("_source_url", "_ingest_timestamp", "data_processamento")


In [0]:
df_registro_classe_cvm = df_registro_classe_cvm.select(
    # 1. Chaves de Identificação
    f.col('ID_Registro_Fundo').cast(t.IntegerType()).alias('id_registro_fundo'),
    f.col('ID_Registro_Classe').cast(t.IntegerType()).alias('id_registro_classe'),
    f.col('CNPJ_Classe').cast(t.StringType()).alias('cnpj_classe'),
    f.col('Codigo_CVM').cast(t.IntegerType()).alias('codigo_cvm'),
    
    # 2. Datas e Status
    f.col('Data_Registro').cast(t.DateType()).alias('data_registro'),
    f.col('Data_Constituicao').cast(t.DateType()).alias('data_constituicao'),
    f.col('Data_Inicio').cast(t.DateType()).alias('data_inicio'),
    f.col('Situacao').cast(t.StringType()).alias('situacao'),
    f.col('Data_Inicio_Situacao').cast(t.DateType()).alias('data_inicio_situacao'),
    
    # 3. Características e Estrutura da Classe
    f.col('Denominacao_Social').cast(t.StringType()).alias('denominacao_social'),
    f.col('Tipo_Classe').cast(t.StringType()).alias('tipo_classe'),
    f.col('Classificacao').cast(t.StringType()).alias('classificacao'),
    f.col('Indicador_Desempenho').cast(t.StringType()).alias('indicador_desempenho'),
    f.col('Classe_Cotas').cast(t.StringType()).alias('classe_cotas'),
    f.col('Classificacao_Anbima').cast(t.StringType()).alias('classificacao_anbima'),
    f.col('Tributacao_Longo_Prazo').cast(t.StringType()).alias('tributacao_longo_prazo'),
    f.col('Entidade_Investimento').cast(t.StringType()).alias('entidade_investimento'),
    f.col('Permitido_Aplicacao_CemPorCento_Exterior').cast(t.StringType()).alias('permitido_aplicacao_cemporcento_exterior'),
    f.col('Classe_ESG').cast(t.StringType()).alias('classe_esg'),
    f.col('Forma_Condominio').cast(t.StringType()).alias('forma_condominio'),
    f.col('Exclusivo').cast(t.StringType()).alias('exclusivo'),
    f.col('Publico_Alvo').cast(t.StringType()).alias('publico_alvo'),
    
    # 4. Dados Patrimoniais
    f.col('Patrimonio_Liquido').cast(t.DecimalType(25, 2)).alias('patrimonio_liquido'),
    f.col('Data_Patrimonio_Liquido').cast(t.DateType()).alias('data_patrimonio_liquido'),
    
    # 5. Prestadores de Serviço
    f.col('CNPJ_Auditor').cast(t.StringType()).alias('cnpj_auditor'),
    f.col('Auditor').cast(t.StringType()).alias('auditor'),
    f.col('CNPJ_Custodiante').cast(t.StringType()).alias('cnpj_custodiante'),
    f.col('Custodiante').cast(t.StringType()).alias('custodiante'),
    f.col('CNPJ_Controlador').cast(t.StringType()).alias('cnpj_controlador'),
    f.col('Controlador').cast(t.StringType()).alias('controlador')
)

### 1.2 Salvar na camada Silver

In [0]:
# Definindo as chaves estrangeiras 
chave_negocio = ["cnpj_classe"]

PipelineConfig.upsert_silver(
    spark=spark, 
    df_novo=df_registro_classe_cvm, 
    tabela_destino=SILVER_PATH, 
    chave_negocio=chave_negocio
    )